# Tests de integración de `import_traces_from_dataframe()` y `validate_traces()`

Notebook orientado a verificar el comportamiento end-to-end de OP-14 `import traces` y OP-15 `validate traces`, con foco en:

- contrato observable de las funciones públicas,
- reporte + metadata + events,
- política de `strict`,
- bordes fatales vs. degradados,
- y cobertura razonable de dtypes/constraints en `TraceSchema`.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if not (REPO_ROOT / "pylondrina").exists():
    REPO_ROOT = Path(".").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:

- imports generales,
- imports del módulo,
- helpers de testing,
- schema base de traces,
- y fixtures reutilizables con datasets de puntos no triviales.

In [2]:
import json
from copy import deepcopy
from pprint import pprint

import numpy as np
import pandas as pd

### 0.1 Imports del módulo

In [3]:
from pylondrina.schema import FieldSpec, TraceSchema
from pylondrina.datasets import TraceDataset
from pylondrina.reports import ImportReport, ConsistencyReport
from pylondrina.errors import ImportError as PylondrinaImportError
from pylondrina.errors import ValidationError, SchemaError
from pylondrina.importing_traces import ImportTraceOptions, import_traces_from_dataframe
from pylondrina.validation_traces import TraceValidationOptions, validate_traces

### 0.2 Helpers de testing reutilizables

In [4]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, default=str)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [issue.code if hasattr(issue, "code") else issue.get("code") for issue in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def clone_tracedataset(traces: TraceDataset) -> TraceDataset:
    return TraceDataset(
        data=traces.data.copy(deep=True),
        schema=traces.schema,
        provenance=deepcopy(traces.provenance),
        metadata=deepcopy(traces.metadata),
    )

### 0.3 Fixtures base de traces

Qué contiene:

- un `TraceSchema` base con varios `FieldSpec`,
- helpers de construcción de columnas/campos,
- y dos fixtures de puntos crudos con varias filas y varias columnas.

In [5]:
def make_trace_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints or {},
    )


def make_trace_schema() -> TraceSchema:
    fields = {
        "point_id": make_trace_field(
            "point_id",
            "string",
            required=True,
            constraints={"unique": True, "length": {"min": 2, "max": 20}},
        ),
        "user_id": make_trace_field(
            "user_id",
            "string",
            required=True,
            constraints={"pattern": r"^u_\d{2}$", "length": {"min": 4, "max": 4}},
        ),
        "time_utc": make_trace_field(
            "time_utc",
            "datetime",
            required=True,
            constraints={"datetime": {"allow_naive": True}},
        ),
        "latitude": make_trace_field(
            "latitude",
            "float",
            required=True,
            constraints={"range": {"min": -90.0, "max": 90.0}},
        ),
        "longitude": make_trace_field(
            "longitude",
            "float",
            required=True,
            constraints={"range": {"min": -180.0, "max": 180.0}},
        ),
        "visit_code": make_trace_field(
            "visit_code",
            "string",
            constraints={"pattern": r"^v\d{3}$", "length": {"min": 4, "max": 4}},
        ),
        "battery_pct": make_trace_field(
            "battery_pct",
            "int",
            constraints={"range": {"min": 0, "max": 100}},
        ),
        "speed_mps": make_trace_field(
            "speed_mps",
            "float",
            constraints={"range": {"min": 0.0, "max": 80.0}},
        ),
        "is_home": make_trace_field(
            "is_home",
            "bool",
            constraints={"nullable": False},
        ),
    }

    return TraceSchema(
        version="golondrina-trace-1.1-test",
        fields=fields,
        required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
        crs="EPSG:4326",
        timezone="UTC",
    )


def make_raw_points_df(n_users: int = 3, points_per_user: int = 4) -> pd.DataFrame:
    rows = []
    for u in range(n_users):
        for j in range(points_per_user):
            i = u * points_per_user + j
            rows.append(
                {
                    "uid": f"u_{u+1:02d}",
                    "ts_local": f"2026-01-{u+1:02d} {8 + j:02d}:{(i % 6):02d}:00",
                    "lat_src": -33.45 - 0.002 * i,
                    "lon_src": -70.66 - 0.002 * i,
                    "visit_code": f"v{i:03d}",
                    "battery_pct": 95 - i,
                    "speed_mps": float(j) + 0.25,
                    "is_home": "true" if j % 2 == 0 else "false",
                    "device_vendor": "acme",
                    "poi_name": f"poi_{i}",
                    "sample_weight": round(1.0 + 0.1 * j, 2),
                }
            )
    return pd.DataFrame(rows)


TRACE_SCHEMA_BASE = make_trace_schema()

FIELD_CORRESPONDENCE_BASE = {
    "user_id": "uid",
    "time_utc": "ts_local",
    "latitude": "lat_src",
    "longitude": "lon_src",
}

raw_points_df = make_raw_points_df(n_users=3, points_per_user=5)
raw_points_df_large = make_raw_points_df(n_users=4, points_per_user=6)

### Test 1 - caso principal correcto con dataset más grande

Qué prueba: el camino principal correcto de OP-14 sobre un dataset más rico, usando `field_correspondence`, `source_timezone`, preservación de extras, generación de `point_id`, metadata mínima, evento `import_traces` y `ImportReport` coherente.

In [8]:
dataset_import_large, report_import_large = import_traces_from_dataframe(
    raw_points_df_large,
    TRACE_SCHEMA_BASE,
    source_name="integration_raw_points_large",
    options=ImportTraceOptions(source_timezone="America/Santiago"),
    field_correspondence=FIELD_CORRESPONDENCE_BASE,
    provenance={"stage": "integration", "batch": "large_case"},
)
display(raw_points_df_large.head())
display(dataset_import_large.data.head())
display(report_import_large.issues)

assert isinstance(dataset_import_large, TraceDataset)
assert isinstance(report_import_large, ImportReport)

assert report_import_large.ok is True
assert report_import_large.summary["rows_in"] == len(raw_points_df_large)
assert report_import_large.summary["rows_out"] == len(raw_points_df_large)
assert report_import_large.summary["n_fields_mapped"] == 4
assert report_import_large.summary["point_id_generated"] is True

assert dataset_import_large.metadata["is_validated"] is False
assert dataset_import_large.metadata["point_id_generated"] is True
assert dataset_import_large.metadata["events"][-1]["op"] == "import_traces"
assert dataset_import_large.data.columns[:5].tolist() == [
    "point_id",
    "user_id",
    "time_utc",
    "latitude",
    "longitude",
]

assert "device_vendor" in dataset_import_large.data.columns
assert "poi_name" in dataset_import_large.data.columns

assert_issue_present(report_import_large.issues, "IMP.CORE.POINT_ID_GENERATED")
assert_json_safe(dataset_import_large.metadata, "dataset_import_large.metadata")
assert_json_safe(report_import_large.summary, "report_import_large.summary")

show_ok("OP-14 / Test 1 - import principal correcto")

,uid,ts_local,lat_src,lon_src,visit_code,battery_pct,speed_mps,is_home,device_vendor,poi_name,sample_weight
0,u_01,2026-01-01 08:00:00,-33.450,-70.660,v000,95,0.25,true,acme,poi_0,1.0
1,u_01,2026-01-01 09:01:00,-33.452,-70.662,v001,94,1.25,false,acme,poi_1,1.1
2,u_01,2026-01-01 10:02:00,-33.454,-70.664,v002,93,2.25,true,acme,poi_2,1.2
3,u_01,2026-01-01 11:03:00,-33.456,-70.666,v003,92,3.25,false,acme,poi_3,1.3
4,u_01,2026-01-01 12:04:00,-33.458,-70.668,v004,91,4.25,true,acme,poi_4,1.4


,point_id,user_id,time_utc,latitude,longitude,visit_code,battery_pct,speed_mps,is_home,device_vendor,poi_name,sample_weight
0,p0,u_01,2026-01-01 11:00:00,-33.450,-70.660,v000,95,0.25,true,acme,poi_0,1.0
1,p1,u_01,2026-01-01 12:01:00,-33.452,-70.662,v001,94,1.25,false,acme,poi_1,1.1
2,p2,u_01,2026-01-01 13:02:00,-33.454,-70.664,v002,93,2.25,true,acme,poi_2,1.2
3,p3,u_01,2026-01-01 14:03:00,-33.456,-70.666,v003,92,3.25,false,acme,poi_3,1.3
4,p4,u_01,2026-01-01 15:04:00,-33.458,-70.668,v004,91,4.25,true,acme,poi_4,1.4


[Issue(level='info', code='IMP.CORE.POINT_ID_GENERATED', message='No se encontró point_id alcanzable; se generó automáticamente una columna técnica secuencial.', field='point_id', source_field=None, row_count=None, details={'field': 'point_id', 'insert_position': 0, 'rows_out': 24, 'action': 'generate_point_id'})]

OK - OP-14 / Test 1 - import principal correcto


### Test 2 - `selected_fields=[]` conserva solo el núcleo canónico

Qué prueba: la semántica especial de `selected_fields=[]`. Debe conservar solo `{point_id, user_id, time_utc, latitude, longitude}`, aunque existan muchos campos extra en la fuente.

In [11]:
dataset_import_core_only, report_import_core_only = import_traces_from_dataframe(
    raw_points_df,
    TRACE_SCHEMA_BASE,
    options=ImportTraceOptions(
        selected_fields=[],
        keep_extra_fields=True,
        source_timezone="UTC",
    ),
    field_correspondence=FIELD_CORRESPONDENCE_BASE,
)
display(raw_points_df.head())
display(dataset_import_core_only.data.head())
display(report_import_core_only.issues)

assert dataset_import_core_only.data.columns.tolist() == [
    "point_id",
    "user_id",
    "time_utc",
    "latitude",
    "longitude",
]

assert_issue_present(report_import_core_only.issues, "IMP.OPTIONS.EMPTY_SELECTED_FIELDS")
assert_issue_present(report_import_core_only.issues, "IMP.OPTIONS.EXTRA_FIELDS_DROPPED")
assert dataset_import_core_only.metadata["is_validated"] is False

show_ok("OP-14 / Test 2 - selected_fields vacío")

,uid,ts_local,lat_src,lon_src,visit_code,battery_pct,speed_mps,is_home,device_vendor,poi_name,sample_weight
0,u_01,2026-01-01 08:00:00,-33.450,-70.660,v000,95,0.25,true,acme,poi_0,1.0
1,u_01,2026-01-01 09:01:00,-33.452,-70.662,v001,94,1.25,false,acme,poi_1,1.1
2,u_01,2026-01-01 10:02:00,-33.454,-70.664,v002,93,2.25,true,acme,poi_2,1.2
3,u_01,2026-01-01 11:03:00,-33.456,-70.666,v003,92,3.25,false,acme,poi_3,1.3
4,u_01,2026-01-01 12:04:00,-33.458,-70.668,v004,91,4.25,true,acme,poi_4,1.4


,point_id,user_id,time_utc,latitude,longitude
0,p0,u_01,2026-01-01 08:00:00,-33.450,-70.660
1,p1,u_01,2026-01-01 09:01:00,-33.452,-70.662
2,p2,u_01,2026-01-01 10:02:00,-33.454,-70.664
3,p3,u_01,2026-01-01 11:03:00,-33.456,-70.666
4,p4,u_01,2026-01-01 12:04:00,-33.458,-70.668


[Issue(level='info', code='IMP.OPTIONS.EMPTY_SELECTED_FIELDS', message='selected_fields está vacío; se conservará únicamente el núcleo canónico obligatorio de traces.', field=None, source_field=None, row_count=None, details={'selected_fields': [], 'effective_selected_fields': ['point_id', 'user_id', 'time_utc', 'latitude', 'longitude'], 'fallback': 'canonical_core_only'}),
 Issue(level='info', code='IMP.OPTIONS.EXTRA_FIELDS_DROPPED', message='Se descartaron columnas extra porque la política efectiva de selección/conservación no las admite (n=7).', field=None, source_field=None, row_count=None, details={'keep_extra_fields': True, 'selected_fields': [], 'n_dropped': 7, 'dropped_columns_sample': ['visit_code', 'battery_pct', 'speed_mps', 'is_home', 'device_vendor', 'poi_name', 'sample_weight'], 'dropped_columns_total': 7, 'action': 'drop_extra_fields'}),
 Issue(level='info', code='IMP.CORE.POINT_ID_GENERATED', message='No se encontró point_id alcanzable; se generó automáticamente una colu

OK - OP-14 / Test 2 - selected_fields vacío


### Test 3 - selección fina con unknown selected field y `keep_extra_fields=False`

Qué prueba: selección explícita de algunos campos válidos, omisión recuperable de nombres inexistentes y descarte de extras por política efectiva.

In [12]:
dataset_import_selected, report_import_selected = import_traces_from_dataframe(
    raw_points_df,
    TRACE_SCHEMA_BASE,
    options=ImportTraceOptions(
        keep_extra_fields=False,
        selected_fields=["visit_code", "battery_pct", "missing_extra"],
        source_timezone="UTC",
    ),
    field_correspondence=FIELD_CORRESPONDENCE_BASE,
)
display(raw_points_df.head())
display(dataset_import_selected.data.head())
display(report_import_selected.issues)

assert dataset_import_selected.data.columns.tolist() == [
    "point_id",
    "user_id",
    "time_utc",
    "latitude",
    "longitude",
    "visit_code",
    "battery_pct",
]

assert_issue_present(report_import_selected.issues, "IMP.OPTIONS.SELECTED_FIELDS_UNKNOWN")
assert_issue_present(report_import_selected.issues, "IMP.OPTIONS.EXTRA_FIELDS_DROPPED")
assert dataset_import_selected.metadata["is_validated"] is False

show_ok("OP-14 / Test 3 - selected_fields con unknowns")

,uid,ts_local,lat_src,lon_src,visit_code,battery_pct,speed_mps,is_home,device_vendor,poi_name,sample_weight
0,u_01,2026-01-01 08:00:00,-33.450,-70.660,v000,95,0.25,true,acme,poi_0,1.0
1,u_01,2026-01-01 09:01:00,-33.452,-70.662,v001,94,1.25,false,acme,poi_1,1.1
2,u_01,2026-01-01 10:02:00,-33.454,-70.664,v002,93,2.25,true,acme,poi_2,1.2
3,u_01,2026-01-01 11:03:00,-33.456,-70.666,v003,92,3.25,false,acme,poi_3,1.3
4,u_01,2026-01-01 12:04:00,-33.458,-70.668,v004,91,4.25,true,acme,poi_4,1.4


,point_id,user_id,time_utc,latitude,longitude,visit_code,battery_pct
0,p0,u_01,2026-01-01 08:00:00,-33.450,-70.660,v000,95
1,p1,u_01,2026-01-01 09:01:00,-33.452,-70.662,v001,94
2,p2,u_01,2026-01-01 10:02:00,-33.454,-70.664,v002,93
3,p3,u_01,2026-01-01 11:03:00,-33.456,-70.666,v003,92
4,p4,u_01,2026-01-01 12:04:00,-33.458,-70.668,v004,91


[Issue(level='warning', code='IMP.OPTIONS.SELECTED_FIELDS_UNKNOWN', message='selected_fields contiene nombres que no existen tras el mapeo efectivo; se omitirán (n=1).', field=None, source_field=None, row_count=None, details={'selected_fields': ['visit_code', 'battery_pct', 'missing_extra'], 'unknown_fields': ['missing_extra'], 'n_unknown': 1, 'available_columns_sample': ['user_id', 'time_utc', 'latitude', 'longitude', 'visit_code', 'battery_pct', 'speed_mps', 'is_home', 'device_vendor', 'poi_name', 'sample_weight'], 'available_columns_total': 11, 'action': 'omit_unknown_selected_fields'}),
 Issue(level='info', code='IMP.OPTIONS.EXTRA_FIELDS_DROPPED', message='Se descartaron columnas extra porque la política efectiva de selección/conservación no las admite (n=5).', field=None, source_field=None, row_count=None, details={'keep_extra_fields': False, 'selected_fields': ['visit_code', 'battery_pct', 'missing_extra'], 'n_dropped': 5, 'dropped_columns_sample': ['speed_mps', 'is_home', 'devic

OK - OP-14 / Test 3 - selected_fields con unknowns


### Test 4 - provenance inválido como degradación recuperable

Qué prueba: un caso degradado en que `provenance` no es un mapping usable. La operación no debe abortar; debe emitir issue recuperable y dejar `provenance` vacío en el `TraceDataset` de salida.

In [15]:
dataset_import_bad_prov, report_import_bad_prov = import_traces_from_dataframe(
    raw_points_df,
    TRACE_SCHEMA_BASE,
    options=ImportTraceOptions(source_timezone="UTC"),
    field_correspondence=FIELD_CORRESPONDENCE_BASE,
    provenance=["this", "is", "not", "a", "mapping"],
)
display(raw_points_df.head())
display(dataset_import_bad_prov.data.head())
display(report_import_bad_prov.issues)

assert_issue_present(report_import_bad_prov.issues, "IMP.PROVENANCE.INVALID_STRUCTURE")
assert dataset_import_bad_prov.provenance == {}
assert dataset_import_bad_prov.metadata["events"][-1]["op"] == "import_traces"

show_ok("OP-14 / Test 4 - provenance inválido degradado")

,uid,ts_local,lat_src,lon_src,visit_code,battery_pct,speed_mps,is_home,device_vendor,poi_name,sample_weight
0,u_01,2026-01-01 08:00:00,-33.450,-70.660,v000,95,0.25,true,acme,poi_0,1.0
1,u_01,2026-01-01 09:01:00,-33.452,-70.662,v001,94,1.25,false,acme,poi_1,1.1
2,u_01,2026-01-01 10:02:00,-33.454,-70.664,v002,93,2.25,true,acme,poi_2,1.2
3,u_01,2026-01-01 11:03:00,-33.456,-70.666,v003,92,3.25,false,acme,poi_3,1.3
4,u_01,2026-01-01 12:04:00,-33.458,-70.668,v004,91,4.25,true,acme,poi_4,1.4


,point_id,user_id,time_utc,latitude,longitude,visit_code,battery_pct,speed_mps,is_home,device_vendor,poi_name,sample_weight
0,p0,u_01,2026-01-01 08:00:00,-33.450,-70.660,v000,95,0.25,true,acme,poi_0,1.0
1,p1,u_01,2026-01-01 09:01:00,-33.452,-70.662,v001,94,1.25,false,acme,poi_1,1.1
2,p2,u_01,2026-01-01 10:02:00,-33.454,-70.664,v002,93,2.25,true,acme,poi_2,1.2
3,p3,u_01,2026-01-01 11:03:00,-33.456,-70.666,v003,92,3.25,false,acme,poi_3,1.3
4,p4,u_01,2026-01-01 12:04:00,-33.458,-70.668,v004,91,4.25,true,acme,poi_4,1.4


[Issue(level='info', code='IMP.CORE.POINT_ID_GENERATED', message='No se encontró point_id alcanzable; se generó automáticamente una columna técnica secuencial.', field='point_id', source_field=None, row_count=None, details={'field': 'point_id', 'insert_position': 0, 'rows_out': 15, 'action': 'generate_point_id'}),
 Issue(level='warning', code='IMP.PROVENANCE.INVALID_STRUCTURE', message='El bloque provenance no es usable o no es JSON-safe; se omitirá del TraceDataset resultante.', field=None, source_field=None, row_count=None, details={'received_type': 'list', 'reason': 'not_mapping', 'exception_type': None, 'exception_message': None, 'action': 'drop_invalid_provenance'})]

OK - OP-14 / Test 4 - provenance inválido degradado


### Test 5 - fatal por mínimo canónico inalcanzable

Qué prueba: una falla realmente fatal de OP-14. Aquí el input no puede alcanzar `latitude`, por lo que la operación debe abortar con `ImportError` y code `IMP.CORE.MINIMUM_FIELDS_UNREACHABLE`.

In [16]:
raw_points_missing_lat = raw_points_df.drop(columns=["lat_src"])

try:
    import_traces_from_dataframe(
        raw_points_missing_lat,
        TRACE_SCHEMA_BASE,
        options=ImportTraceOptions(source_timezone="UTC"),
        field_correspondence=FIELD_CORRESPONDENCE_BASE,
    )
    raise AssertionError("Se esperaba ImportError por mínimo canónico inalcanzable.")
except PylondrinaImportError as exc:
    assert exc.code == "IMP.CORE.MINIMUM_FIELDS_UNREACHABLE"
    display(exc.issues)

show_ok("OP-14 / Test 5 - fatal por mínimo inalcanzable")

(Issue(level='warning', code='MAP.FIELDS.SOURCE_COLUMN_NOT_FOUND', message="La columna fuente 'lat_src' declarada para el campo canónico 'latitude' no existe en el DataFrame de entrada.", field='latitude', source_field='lat_src', row_count=None, details={'field': 'latitude', 'source_field': 'lat_src', 'available_columns_sample': ['uid', 'ts_local', 'lon_src', 'visit_code', 'battery_pct', 'speed_mps', 'is_home', 'device_vendor', 'poi_name', 'sample_weight'], 'available_columns_total': 10, 'action': 'skip_mapping_entry'}),
 Issue(level='info', code='IMP.CORE.POINT_ID_GENERATED', message='No se encontró point_id alcanzable; se generó automáticamente una columna técnica secuencial.', field='point_id', source_field=None, row_count=None, details={'field': 'point_id', 'insert_position': 0, 'rows_out': 15, 'action': 'generate_point_id'}),
 Issue(level='error', code='IMP.CORE.MINIMUM_FIELDS_UNREACHABLE', message="No fue posible materializar el núcleo canónico mínimo de traces: ['latitude'].", f

OK - OP-14 / Test 5 - fatal por mínimo inalcanzable


### Test 6 - caso principal correcto end-to-end: import traces -> validate traces

Qué prueba: el camino feliz completo del bloque traces. Primero se importa un dataset relativamente rico y luego se valida con la función pública. Se revisa `ConsistencyReport.summary`, evento `validate_traces`, `metadata["is_validated"]` y no mutación tabular.

In [19]:
traces_validate_ok = clone_tracedataset(dataset_import_large)
data_before_validate_ok = traces_validate_ok.data.copy(deep=True)

report_validate_ok = validate_traces(traces_validate_ok)
display(report_validate_ok.issues)
display(report_validate_ok.summary)

assert isinstance(report_validate_ok, ConsistencyReport)
assert report_validate_ok.summary["ok"] is True
assert report_validate_ok.summary["n_errors"] == 0
assert report_validate_ok.summary["n_warnings"] == 0
assert traces_validate_ok.metadata["is_validated"] is True
assert traces_validate_ok.metadata["events"][-1]["op"] == "validate_traces"
assert traces_validate_ok.metadata["events"][-1]["summary"] == report_validate_ok.summary
assert traces_validate_ok.data.equals(data_before_validate_ok)

assert_json_safe(report_validate_ok.summary, "report_validate_ok.summary")
assert_json_safe(traces_validate_ok.metadata["events"][-1], "validate_traces event")

show_ok("OP-15 / Test 6 - validate principal correcto end-to-end")

[]

{'ok': True,
 'n_rows': 24,
 'n_issues': 0,
 'n_errors': 0,
 'n_warnings': 0,
 'n_info': 0,
 'counts_by_level': {'error': 0, 'warning': 0, 'info': 0},
 'counts_by_code': {},
 'checked_fields': ['point_id',
  'user_id',
  'time_utc',
  'latitude',
  'longitude',
  'visit_code',
  'battery_pct',
  'speed_mps',
  'is_home'],
 'checks_executed': {'required_fields': True,
  'types_and_formats': True,
  'constraints': True,
  'monotonic_time_per_user': True},
 'schema_version': 'golondrina-trace-1.1-test'}

OK - OP-15 / Test 6 - validate principal correcto end-to-end


### Test 7 - required column missing con `strict=False`

Qué prueba: un error de conformidad por datos/estructura observable, no fatal de schema/config. Debe retornar `ConsistencyReport`, dejar evento, marcar `is_validated=False` y no lanzar excepción si `strict=False`.

In [21]:
traces_missing_required = clone_tracedataset(dataset_import_large)
traces_missing_required.data = traces_missing_required.data.drop(columns=["user_id"])

report_missing_required = validate_traces(traces_missing_required)
display(report_missing_required.issues)

assert report_missing_required.summary["ok"] is False
assert report_missing_required.summary["n_errors"] >= 1
assert_issue_present(report_missing_required.issues, "VAL.REQUIRED.MISSING_COLUMN")
assert traces_missing_required.metadata["is_validated"] is False
assert traces_missing_required.metadata["events"][-1]["op"] == "validate_traces"

show_ok("OP-15 / Test 7 - missing required column")

[Issue(level='error', code='VAL.REQUIRED.MISSING_COLUMN', message="Falta el campo obligatorio 'user_id' en traces.data; no se cumple el mínimo requerido de traces.", field='user_id', source_field=None, row_count=None, details={'check': 'required_fields', 'field': 'user_id', 'required_fields': ['point_id', 'user_id', 'time_utc', 'latitude', 'longitude'], 'present_fields': ['point_id', 'time_utc', 'latitude', 'longitude', 'visit_code', 'battery_pct', 'speed_mps', 'is_home', 'device_vendor', 'poi_name', 'sample_weight'], 'action': 'report_error'})]

OK - OP-15 / Test 7 - missing required column


### Test 8 - mezcla de errores de tipos/formatos y constraints

Qué prueba: varios hallazgos integrados en una sola corrida pública de `validate_traces`, cubriendo datetime, float, bool, pattern/length y range.

In [23]:
traces_mixed_errors = clone_tracedataset(dataset_import_large)

# Se pasan a object solo las columnas que vamos a “romper” para evitar warnings de pandas.
traces_mixed_errors.data["time_utc"] = traces_mixed_errors.data["time_utc"].astype("object")
traces_mixed_errors.data["latitude"] = traces_mixed_errors.data["latitude"].astype("object")

traces_mixed_errors.data.loc[1, "time_utc"] = "not-a-time"
traces_mixed_errors.data.loc[2, "latitude"] = "north"
traces_mixed_errors.data.loc[3, "visit_code"] = "BAD"
traces_mixed_errors.data.loc[4, "battery_pct"] = 150
traces_mixed_errors.data.loc[5, "is_home"] = "maybe"

report_mixed_errors = validate_traces(traces_mixed_errors)
mixed_codes = get_issue_codes(report_mixed_errors.issues)

display(report_mixed_errors.issues)

assert report_mixed_errors.summary["ok"] is False
assert "VAL.TYPES.UNPARSEABLE_VALUE" in mixed_codes
assert "VAL.CONSTRAINTS.VIOLATION" in mixed_codes
assert report_mixed_errors.summary["n_errors"] >= 5
assert traces_mixed_errors.metadata["is_validated"] is False

show_ok("OP-15 / Test 8 - mixed types and constraints")

[Issue(level='error', code='VAL.TYPES.UNPARSEABLE_VALUE', message="Se detectaron valores no parseables o no interpretables en 'time_utc'.", field='time_utc', source_field=None, row_count=1, details={'check': 'types_and_formats', 'n_rows_total': 24, 'n_violations': 1, 'row_indices_sample': [1], 'sample_rows': [{'point_id': 'p1', 'user_id': 'u_01', 'time_utc': 'not-a-time', 'latitude': -33.452000000000005, 'longitude': -70.66199999999999, 'visit_code': 'v001', 'battery_pct': 94, 'speed_mps': 1.25, 'is_home': 'false', 'device_vendor': 'acme', 'poi_name': 'poi_1', 'sample_weight': 1.1}], 'field': 'time_utc', 'expected': 'datetime', 'raw_values_sample': ['not-a-time'], 'action': 'report_error'}),
 Issue(level='error', code='VAL.TYPES.UNPARSEABLE_VALUE', message="Se detectaron valores no parseables o no interpretables en 'latitude'.", field='latitude', source_field=None, row_count=1, details={'check': 'types_and_formats', 'n_rows_total': 24, 'n_violations': 1, 'row_indices_sample': [2], 'sam

OK - OP-15 / Test 8 - mixed types and constraints


### Test 9 - nullable efectivo y unique en una misma corrida

Qué prueba: dos constraints declarativas distintas dentro de `validate_traces`:
- `nullable=False` en un campo no required (`is_home`)
- `unique=True` en `point_id`

In [24]:
traces_nullable_unique = clone_tracedataset(dataset_import_large)

traces_nullable_unique.data.loc[0, "is_home"] = None
traces_nullable_unique.data.loc[1, "point_id"] = traces_nullable_unique.data.loc[0, "point_id"]

report_nullable_unique = validate_traces(traces_nullable_unique)

assert report_nullable_unique.summary["ok"] is False
assert report_nullable_unique.summary["counts_by_code"].get("VAL.CONSTRAINTS.VIOLATION", 0) >= 2
assert traces_nullable_unique.metadata["is_validated"] is False

show_ok("OP-15 / Test 9 - nullable efectivo + unique")

OK - OP-15 / Test 9 - nullable efectivo + unique


### Test 10 - monotonicidad temporal por usuario rota, pero solo warning

Qué prueba: la semántica cerrada de `validate_monotonic_time_per_user`. Debe operar sobre el orden observado del dataframe, emitir warning y mantener `ok=True` e `is_validated=True` si no hay errores.

In [26]:
traces_non_monotonic = clone_tracedataset(dataset_import_large)

user_mask = traces_non_monotonic.data["user_id"] == "u_01"
user_index = traces_non_monotonic.data.index[user_mask].tolist()

traces_non_monotonic.data.loc[user_index[3], "time_utc"] = (
    traces_non_monotonic.data.loc[user_index[0], "time_utc"] - pd.Timedelta(hours=1)
)

report_non_monotonic = validate_traces(traces_non_monotonic)
display(report_missing_required.issues)

assert report_non_monotonic.summary["ok"] is True
assert report_non_monotonic.summary["n_warnings"] >= 1
assert_issue_present(report_non_monotonic.issues, "VAL.TEMPORAL.NON_MONOTONIC_TIME")
assert traces_non_monotonic.metadata["is_validated"] is True

show_ok("OP-15 / Test 10 - monotonicidad temporal warning")

[Issue(level='error', code='VAL.REQUIRED.MISSING_COLUMN', message="Falta el campo obligatorio 'user_id' en traces.data; no se cumple el mínimo requerido de traces.", field='user_id', source_field=None, row_count=None, details={'check': 'required_fields', 'field': 'user_id', 'required_fields': ['point_id', 'user_id', 'time_utc', 'latitude', 'longitude'], 'present_fields': ['point_id', 'time_utc', 'latitude', 'longitude', 'visit_code', 'battery_pct', 'speed_mps', 'is_home', 'device_vendor', 'poi_name', 'sample_weight'], 'action': 'report_error'})]

OK - OP-15 / Test 10 - monotonicidad temporal warning


### Test 11 - política clave: `strict=True` con error de datos

Qué prueba: la política central de OP-15. Con `strict=True` y error de datos, primero debe quedar la evidencia construida y luego recién levantarse `ValidationError`.

In [27]:
traces_strict_error = clone_tracedataset(dataset_import_large)
traces_strict_error.data.loc[0, "battery_pct"] = 500

try:
    validate_traces(
        traces_strict_error,
        options=TraceValidationOptions(strict=True),
    )
    raise AssertionError("Se esperaba ValidationError con strict=True.")
except ValidationError as exc:
    assert exc.code == "VAL.CONSTRAINTS.VIOLATION"
    assert traces_strict_error.metadata["is_validated"] is False
    assert traces_strict_error.metadata["events"][-1]["op"] == "validate_traces"
    display(exc.issues)

show_ok("OP-15 / Test 11 - strict=True con evidencia previa")

(Issue(level='error', code='VAL.CONSTRAINTS.VIOLATION', message="Se detectaron violaciones de la constraint 'range' en el campo 'battery_pct'.", field='battery_pct', source_field=None, row_count=1, details={'check': 'constraints', 'n_rows_total': 24, 'n_violations': 1, 'row_indices_sample': [0], 'sample_rows': [{'point_id': 'p0', 'user_id': 'u_01', 'time_utc': '2026-01-01T11:00:00', 'latitude': -33.45, 'longitude': -70.66, 'visit_code': 'v000', 'battery_pct': 500, 'speed_mps': 0.25, 'is_home': 'true', 'device_vendor': 'acme', 'poi_name': 'poi_0', 'sample_weight': 1.0}], 'field': 'battery_pct', 'constraint': 'range', 'expected': {'min': 0, 'max': 100}, 'action': 'report_error'}),)

OK - OP-15 / Test 11 - strict=True con evidencia previa


### Test 12 - fatal de schema/config antes de reporte o evento

Qué prueba: una ruta fatal de preflight de schema/config. Aquí el `TraceSchema` trae `categorical`, lo que debe abortar con `SchemaError` antes de construir reporte o tocar metadata.

In [29]:
TRACE_SCHEMA_BAD_CATEGORICAL = TraceSchema(
    version="bad-trace-schema",
    fields={
        "user_id": make_trace_field("user_id", "categorical", required=True),
        "time_utc": make_trace_field("time_utc", "datetime", required=True),
        "latitude": make_trace_field("latitude", "float", required=True),
        "longitude": make_trace_field("longitude", "float", required=True),
    },
    required=["user_id", "time_utc", "latitude", "longitude"],
)

traces_schema_bad = TraceDataset(
    data=dataset_import_large.data.copy(deep=True),
    schema=TRACE_SCHEMA_BAD_CATEGORICAL,
    provenance={},
    metadata={},
)

try:
    validate_traces(traces_schema_bad)
    raise AssertionError("Se esperaba SchemaError por categorical en TraceSchema.")
except SchemaError as exc:
    assert exc.code == "VAL.SCHEMA.CATEGORICAL_NOT_ALLOWED"
    assert traces_schema_bad.metadata == {}
    display(exc.issues)

show_ok("OP-15 / Test 12 - fatal de schema/config")

(Issue(level='error', code='VAL.SCHEMA.CATEGORICAL_NOT_ALLOWED', message="El dtype 'categorical' no está permitido en TraceSchema v1.1 (campo 'user_id').", field='user_id', source_field=None, row_count=None, details={'field': 'user_id', 'dtype': 'categorical', 'allowed_dtypes': ('string', 'int', 'float', 'datetime', 'bool'), 'action': 'abort'}),)

OK - OP-15 / Test 12 - fatal de schema/config
